In [6]:
# Create working directories
!mkdir -p /kaggle/working/Group-Activity-Recognition/utils
!mkdir -p /kaggle/working/Group-Activity-Recognition/outputs

# Install strict requirements
!pip install -q -U albumentations==1.4.24 torchinfo scikit-learn seaborn

In [7]:
%%writefile /kaggle/working/Group-Activity-Recognition/data.py
import cv2
import pickle
import torch
import numpy as np
from pathlib import Path
import sys
from torch.utils.data import Dataset

activities_labels = {
    "person": {class_name.lower(): i for i, class_name in enumerate(["Waiting", "Setting", "Digging", "Falling", "Spiking", "Blocking", "Jumping", "Moving", "Standing"])},
    "group": {class_name: i for i, class_name in enumerate(["r_set", "r_spike", "r-pass", "r_winpoint", "l_winpoint", "l-pass", "l-spike", "l_set"])}
}

class Hierarchical_Group_Activity_DataSet(Dataset):
    def __init__(self, videos_path, annot_path, split, labels, transform=None):
        self.videos_path = Path(videos_path)
        self.transform = transform
        self.labels = labels
        
        with open(annot_path, 'rb') as f:
            videos_annot = pickle.load(f)
            
        self.data = []
        for clip_id in split:
            clip_dirs = videos_annot[str(clip_id)]
            for clip_dir in clip_dirs.keys():
                category = clip_dirs[str(clip_dir)]['category']
                dir_frames = list(clip_dirs[str(clip_dir)]['frame_boxes_dct'].items())
                
                frames_data = []
                for frame_id, boxes in dir_frames:
                    frame_path = f"{videos_path}/{str(clip_id)}/{str(clip_dir)}/{frame_id}.jpg"
                    frames_data.append((frame_path, boxes))
                
                self.data.append({'frames_data': frames_data, 'category': category})      

    def __len__(self):
        return len(self.data)
    
    def extract_person_crops(self, frame, boxes):
        crops, order, person_frame_labels = [], [], []
        for box in boxes:
            x_min, y_min, x_max, y_max = box.box
            x_center = (x_min + x_max) / 2
            person_crop = frame[max(0, y_min):y_max, max(0, x_min):x_max]
            
            if self.transform:
                person_crop = self.transform(image=person_crop)['image']

            person_label = torch.zeros(len(self.labels['person']))
            person_label[self.labels['person'][box.category]] = 1    
            
            crops.append(person_crop)
            order.append(x_center)
            person_frame_labels.append(person_label)
        
        return crops, order, person_frame_labels
                    
    def __getitem__(self, idx):
        sample = self.data[idx]
        group_label = torch.zeros(len(self.labels['group'])) 
        group_label[self.labels['group'][sample['category']]] = 1

        clip, group_labels, person_labels = [], [], []
    
        for frame_path, boxes in sample['frames_data']:
            frame = cv2.imread(frame_path)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # CRITICAL: Convert BGR to RGB
            
            crops, order, frame_labels = self.extract_person_crops(frame, boxes) 
            
            # Sort left-to-right
            sorted_pairs = sorted(zip(order, crops, frame_labels), key=lambda pair: pair[0])
            sorted_crops = [c for _, c, _ in sorted_pairs]
            sorted_labels = [l for _, _, l in sorted_pairs]
                        
            crops = torch.stack(sorted_crops) if sorted_crops else torch.empty(0)
            sorted_labels = torch.stack(sorted_labels) if sorted_labels else torch.empty(0)
            
            person_labels.append(sorted_labels) 
            clip.append(crops)
            group_labels.append(group_label)
    
        # Shape: (Num_Frames, Num_People, C, H, W) -> (Num_People, Num_Frames, C, H, W)
        clip = torch.stack(clip).permute(1, 0, 2, 3, 4) 
        group_labels = torch.stack(group_labels)
        person_labels = torch.stack(person_labels).permute(1, 0, 2) 

        return clip, person_labels, group_labels

def collate_fn(batch):
    clips, person_labels, group_labels = zip(*batch)  
    max_bboxes = 12  
    padded_clips, padded_person_labels = [], []

    for clip, label in zip(clips, person_labels):
        num_bboxes = clip.size(0)
        if num_bboxes < max_bboxes:
            clip_padding = torch.zeros((max_bboxes - num_bboxes, clip.size(1), clip.size(2), clip.size(3), clip.size(4)))
            label_padding = torch.zeros((max_bboxes - num_bboxes, label.size(1), label.size(2)))
            clip = torch.cat((clip, clip_padding), dim=0)
            label = torch.cat((label, label_padding), dim=0)
            
        padded_clips.append(clip)
        padded_person_labels.append(label)
    
    padded_clips = torch.stack(padded_clips)
    padded_person_labels = torch.stack(padded_person_labels)
    group_labels = torch.stack(group_labels)
    
    # Use only the last frame's label for the loss
    group_labels = group_labels[:, -1, :] 
    padded_person_labels = padded_person_labels[:, :, -1, :]  
    
    b, bb, num_class = padded_person_labels.shape
    padded_person_labels = padded_person_labels.view(b * bb, num_class)

    return padded_clips, padded_person_labels, group_labels

class BoxInfo:
    def __init__(self, line):
        words = line.split()
        self.category = words.pop()
        words = [int(string) for string in words]
        self.player_ID = words[0]
        del words[0]

        x1, y1, x2, y2, frame_ID, lost, grouping, generated = words
        self.box = x1, y1, x2, y2
        self.frame_ID = frame_ID
        self.lost = lost
        self.grouping = grouping
        self.generated = generated

sys.modules['boxinfo'] = sys.modules[__name__]        

Overwriting /kaggle/working/Group-Activity-Recognition/data.py


In [8]:
%%writefile /kaggle/working/Group-Activity-Recognition/model.py
import torch
import torch.nn as nn
import torchvision.models as models

class Hierarchical_Group_Activity_Classifer(nn.Module):
    def __init__(self, person_num_classes=9, group_num_classes=8, hidden_size=512, num_layers=2):
        super(Hierarchical_Group_Activity_Classifer, self).__init__()

        # Feature Extractor
        self.feature_extractor = nn.Sequential(
            *list(models.resnet34(weights=models.ResNet34_Weights.DEFAULT).children())[:-1]
        )
        
        # Person-Level LSTM
        self.layer_norm_1 = nn.LayerNorm(512)
        self.lstm_1 = nn.LSTM(
            input_size=512, hidden_size=hidden_size, num_layers=num_layers, 
            batch_first=True, dropout=0.5
        )

        self.fc_1 = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, person_num_classes)
        )

        # Group-Level LSTM
        self.layer_norm_2 = nn.LayerNorm(512)
        self.pool = nn.AdaptiveMaxPool2d((1, 256))
     
        self.lstm_2 = nn.LSTM(
            input_size=512, hidden_size=hidden_size, num_layers=num_layers, 
            batch_first=True, dropout=0.5
        )
        
        self.fc_2 = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, group_num_classes)
        )
    
    def forward(self, x):
        b, bb, seq, c, h, w = x.shape  
        x = x.view(b * bb * seq, c, h, w)  
        x1 = self.feature_extractor(x) 

        x1 = x1.view(b * bb, seq, -1)       
        x1 = self.layer_norm_1(x1)          
        
        # LSTM Fix: Unpack the tuple properly
        x2, (h_1, c_1) = self.lstm_1(x1) 

        y1 = self.fc_1(x2[:, -1, :])  

        x = torch.cat([x1, x2], dim=2) 
        x = x.contiguous()             
       
        x = x.view(b * seq, bb, -1) 
        team_1 = x[:, :6, :]      
        team_2 = x[:, 6:, :]      

        team_1 = self.pool(team_1) 
        team_2 = self.pool(team_2) 
        x = torch.cat([team_1, team_2], dim=1)  
       
        x = x.view(b, seq, -1) 
        x = self.layer_norm_2(x) 
        
        # LSTM Fix: Unpack the tuple properly
        x, (h_2, c_2) = self.lstm_2(x) 

        x = x[:, -1, :]     
        y2 = self.fc_2(x)   
        
        return {'person_output': y1, 'group_output': y2}

Overwriting /kaggle/working/Group-Activity-Recognition/model.py


In [9]:
%%writefile /kaggle/working/Group-Activity-Recognition/train.py
import os
# Silence the Albumentations update warning
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2

from data import Hierarchical_Group_Activity_DataSet, collate_fn, activities_labels
from model import Hierarchical_Group_Activity_Classifer

# Configuration
DATA_DIR = "/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball" 
BATCH_SIZE = 4 # Per GPU (Total = 8)
EPOCHS = 85
LR = 6e-6
WEIGHT_DECAY = 1

train_split = [1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31, 32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54]
val_split = [0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51]

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def train_process(rank, world_size):
    setup_ddp(rank, world_size)
    device = torch.device(f"cuda:{rank}")
    
    # Model Setup
    model = Hierarchical_Group_Activity_Classifer().to(device)
    # === NEW RESUME BLOCK ===
    # Update this path to match exactly where you uploaded the file in Step 1!
    RESUME_PATH = "/kaggle/input/models/abdullahali7/b9/pytorch/default/1/best_b9_lstm_model.pth" 
    if os.path.exists(RESUME_PATH):
        if rank == 0:
            print(f"LOADING PREVIOUS BRAINPOWER FROM: {RESUME_PATH}")
        # map_location safely loads it across the multiple GPUs
        model.load_state_dict(torch.load(RESUME_PATH, map_location=device, weights_only=True))
    # ========================
    model = DDP(model, device_ids=[rank])
    
    # Augmentations
    train_transforms = A.Compose([
        A.Resize(224, 224),
        A.OneOf([A.GaussianBlur((3, 7)), A.ColorJitter(0.2), A.MotionBlur(5)], p=0.55),
        A.HorizontalFlip(p=0.5), # Standard flip is usually fine for Volleyball
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    val_transforms = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    # Datasets & Samplers
    train_dataset = Hierarchical_Group_Activity_DataSet(f"{DATA_DIR}/videos", f"{DATA_DIR}/annot_all.pkl", train_split, activities_labels, train_transforms)
    val_dataset = Hierarchical_Group_Activity_DataSet(f"{DATA_DIR}/videos", f"{DATA_DIR}/annot_all.pkl", val_split, activities_labels, val_transforms)
    
    train_sampler = DistributedSampler(train_dataset)
    val_sampler = DistributedSampler(val_dataset, shuffle=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, collate_fn=collate_fn, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, sampler=val_sampler, collate_fn=collate_fn, num_workers=4, pin_memory=True)

    # Loss & Optimizer
    person_criterion = nn.CrossEntropyLoss(label_smoothing=0.10).to(device)
    group_criterion = nn.CrossEntropyLoss().to(device) 
    
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    # Updated AMP Syntax
    scaler = torch.amp.GradScaler('cuda')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=15) 

    best_acc = 0.0

    # Training Loop
    for epoch in range(EPOCHS):
        train_sampler.set_epoch(epoch)
        model.train()
        total_loss = 0
        
        for batch_idx, (inputs, person_labels, group_labels) in enumerate(train_loader):
            inputs, person_labels, group_labels = inputs.to(device), person_labels.to(device), group_labels.to(device)
            optimizer.zero_grad()
            
            # Updated AMP Syntax
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss_1 = person_criterion(outputs['person_output'], person_labels)
                loss_2 = group_criterion(outputs['group_output'], group_labels)
                
                # The 0.60 Dual-Loss Strategy
                loss = loss_2 + (0.60 * loss_1)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            
            # --- THE HEARTBEAT PRINT ---
            if rank == 0 and batch_idx % 50 == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx}/{len(train_loader)}] | Current Batch Loss: {loss.item():.4f}")
            # ---------------------------
            
        # Validation
        model.eval()
        correct, total = 0, 0
        
        if rank == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Training done. Running Validation...")
            
        with torch.no_grad():
            for inputs, person_labels, group_labels in val_loader:
                inputs, group_labels = inputs.to(device), group_labels.to(device)
                outputs = model(inputs)
                
                _, predicted = outputs['group_output'].max(1)
                _, target_class = group_labels.max(1)
                
                total += group_labels.size(0)
                correct += predicted.eq(target_class).sum().item()
        
        # Aggregate metrics across GPUs
        correct_tensor = torch.tensor(correct).to(device)
        total_tensor = torch.tensor(total).to(device)
        dist.all_reduce(correct_tensor, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
        
        val_acc = 100. * correct_tensor.item() / total_tensor.item()
        scheduler.step(val_acc)
        
        if rank == 0:
            print(f"====> Epoch [{epoch+1}/{EPOCHS}] COMPLETE | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']}")
            if val_acc > best_acc:
                best_acc = val_acc
                
                # --- NEW FOLDER CREATION LINE ---
                os.makedirs("/kaggle/working/outputs", exist_ok=True) 
                
                torch.save(model.module.state_dict(), "/kaggle/working/outputs/best_b9_lstm_model.pth")
                print("--> Saved Best Model!")

    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    if world_size < 2:
        print("Warning: Only 1 GPU detected. DDP requires 2 for T4x2 setup.")
    mp.spawn(train_process, args=(world_size,), nprocs=world_size, join=True)

Overwriting /kaggle/working/Group-Activity-Recognition/train.py


In [10]:
!python /kaggle/working/Group-Activity-Recognition/train.py

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 202MB/s]
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 131MB/s]
LOADING PREVIOUS BRAINPOWER FROM: /kaggle/input/models/abdullahali7/b9/pytorch/default/1/best_b9_lstm_model.pth
Epoch [1/85] | Batch [0/269] | Current Batch Loss: 1.7328
Epoch [1/85] | Batch [50/269] | Current Batch Loss: 0.8744
Epoch [1/85] | Batch [100/269] | Current Batch Loss: 1.3146
Epoch [1/85] | Batch [150/269] | Current Batch Loss: 0.9939
Epoch [1/85] | Batch [200/269] | Current Batch Loss: 1.0617
Epoch [1/85] | Batch [250/269] | Current Batch Loss: 1.3512
Epoch [1/85] Training done. Running Validation...
====> Epoch [1/85] COMPLETE | Train Loss: 1.2045 | Va

In [17]:
%%writefile /kaggle/working/Group-Activity-Recognition/evaluate.py
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Import your custom modules
from data import Hierarchical_Group_Activity_DataSet, collate_fn, activities_labels
from model import Hierarchical_Group_Activity_Classifer

# --- Configuration ---
DATA_DIR = "/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball"
WEIGHTS_PATH = "/kaggle/working/outputs/best_b9_lstm_model.pth" 
OUTPUT_DIR = "/kaggle/working/outputs"
BATCH_SIZE = 8

# Standard Volleyball Dataset Test Split
test_split = [4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47]

def evaluate():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Prepare Data
    # Notice we only use basic transforms here, NO heavy augmentations for testing!
    test_transforms = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    print("Loading test dataset...")
    test_dataset = Hierarchical_Group_Activity_DataSet(
        f"{DATA_DIR}/videos", 
        f"{DATA_DIR}/annot_all.pkl", 
        test_split, 
        activities_labels, 
        test_transforms
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        collate_fn=collate_fn, 
        num_workers=4, 
        pin_memory=True
    )

    # 2. Setup Model
    print("Initializing model and loading weights...")
    model = Hierarchical_Group_Activity_Classifer().to(device)
    
    if os.path.exists(WEIGHTS_PATH):
        # We use weights_only=True for security and to avoid warnings
        model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
        print("--> Weights loaded successfully!")
    else:
        raise FileNotFoundError(f"Could not find weights at {WEIGHTS_PATH}. Please check the path!")

    model.eval()

    # 3. Evaluation Loop
    all_preds = []
    all_targets = []
    
    print("Running evaluation (this may take a few minutes)...")
    with torch.no_grad():
        for batch_idx, (inputs, person_labels, group_labels) in enumerate(test_loader):
            inputs, group_labels = inputs.to(device), group_labels.to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Get the predicted class (highest probability)
            _, predicted = outputs['group_output'].max(1)
            _, target_class = group_labels.max(1)
            
            # Move to CPU and store for scikit-learn
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target_class.cpu().numpy())
            
            if batch_idx % 20 == 0:
                print(f"Processed Batch {batch_idx}/{len(test_loader)}")

    # 4. Metrics & Confusion Matrix
    print("\n" + "="*50)
    print("EVALUATION COMPLETE")
    print("="*50)
    

    class_names = ['r_set', 'r_spike', 'r_pass', 'r_winpoint', 'l_set', 'l_spike', 'l_pass', 'l_winpoint']

    # Print Classification Report (Precision, Recall, F1)
    report = classification_report(all_targets, all_preds, target_names=class_names, zero_division=0)
    print("\nClassification Report:\n")
    print(report)

    # Generate Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # Plotting
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Group Activity Confusion Matrix')
    plt.ylabel('Actual Activity')
    plt.xlabel('Predicted Activity')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save the plot
    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    print(f"\n--> Confusion Matrix saved to: {cm_path}")

if __name__ == "__main__":
    evaluate()

Overwriting /kaggle/working/Group-Activity-Recognition/evaluate.py


In [18]:
!python /kaggle/working/Group-Activity-Recognition/evaluate.py

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Using device: cuda
Loading test dataset...
Initializing model and loading weights...
--> Weights loaded successfully!
Running evaluation (this may take a few minutes)...
Processed Batch 0/168
Processed Batch 20/168
Processed Batch 40/168
Processed Batch 60/168
Processed Batch 80/168
Processed Batch 100/168
Processed Batch 120/168
Processed Batch 140/168
Processed Batch 160/168

EVALUATION COMPLETE

Classification Report:

              precision    recall  f1-score   support

       r_set       0.85      0.70      0.77       192
     r_spike       0.92      0.88      0.90       173
      r_pass       0.75      0.91      0.82       210
  r_winpoint       0.90      0.89     